In [7]:
# Data manipulation and numerical computing
import pandas as pd
import numpy as np

# Visualization (full matplotlib package, its pyplot interface and seaborn)
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
import sklearn

# Web app framework
import streamlit as st

# Public holidays by country
import holidays

In [8]:
# Load the sales training data (path relative to the notebooks folder)
ventas_df = pd.read_csv("../data/raw/entrenamiento/ventas.csv")

# Load the competitors' prices training data
competencia_df = pd.read_csv("../data/raw/entrenamiento/competencia.csv")

In [9]:
# Show the first rows of the sales dataframe
ventas_df.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos
0,2021-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,6,118.36,710.16
1,2021-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,10,136.82,1368.20
2,2021-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,2,84.93,169.86
3,2021-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,2,75.42,150.84
4,2021-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,2,35.87,71.74


In [10]:
# Show the first rows of the competitors' prices dataframe
competencia_df.head()

,fecha,producto_id,Amazon,Decathlon,Deporvillage
0,2021-10-25,PROD_001,82.96,111.88,97.43
1,2021-10-25,PROD_002,112.56,108.61,115.58
2,2021-10-25,PROD_003,79.79,78.44,80.11
3,2021-10-25,PROD_004,72.60,67.29,74.45
4,2021-10-25,PROD_005,37.71,33.60,33.07


In [11]:
# Show the column names of the sales dataframe
ventas_df.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos'],
      dtype='str')

In [12]:
# Show the column names of the competitors' prices dataframe
competencia_df.columns

Index(['fecha', 'producto_id', 'Amazon', 'Decathlon', 'Deporvillage'], dtype='str')

In [13]:
# Show shape, data types and non-null counts of the sales dataframe
print(ventas_df.shape)
ventas_df.info()

# Count null values per column
print(ventas_df.isna().sum())

# Count fully duplicated rows and duplicated (fecha, producto_id) keys
print("Full duplicates:", ventas_df.duplicated().sum())
print("Key duplicates:", ventas_df.duplicated(subset=["fecha", "producto_id"]).sum())

(3552, 10)
<class 'pandas.DataFrame'>
RangeIndex: 3552 entries, 0 to 3551
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   fecha              3552 non-null   str    
 1   producto_id        3552 non-null   str    
 2   nombre             3552 non-null   str    
 3   categoria          3552 non-null   str    
 4   subcategoria       3552 non-null   str    
 5   precio_base        3552 non-null   int64  
 6   es_estrella        3552 non-null   bool   
 7   unidades_vendidas  3552 non-null   int64  
 8   precio_venta       3552 non-null   float64
 9   ingresos           3552 non-null   float64
dtypes: bool(1), float64(2), int64(2), str(5)
memory usage: 467.8 KB
fecha                0
producto_id          0
nombre               0
categoria            0
subcategoria         0
precio_base          0
es_estrella          0
unidades_vendidas    0
precio_venta         0
ingresos             0
dtype: int64
Full 

In [14]:
# Descriptive statistics of the numeric columns
display(ventas_df.describe().T)

# Descriptive statistics of the categorical columns
display(ventas_df.select_dtypes(exclude="number").describe().T)

# Frequency of the categories and subcategories
print(ventas_df["categoria"].value_counts())
print(ventas_df["subcategoria"].value_counts())

,count,mean,std,min,25%,50%,75%,max
precio_base,3552.0,123.125000,165.576753,20.00,48.7500,72.50,118.7500,830.00
unidades_vendidas,3552.0,4.878660,6.311020,1.00,2.0000,3.00,5.0000,85.00
precio_venta,3552.0,121.816546,164.017963,19.00,47.2125,71.81,118.2200,854.22
ingresos,3552.0,605.972323,1079.071192,19.46,131.5350,216.57,639.6375,14508.40


,count,unique,top,freq
fecha,3552,148,2021-10-25,24
producto_id,3552,24,PROD_001,148
nombre,3552,24,Nike Air Zoom Pegasus 40,148
categoria,3552,4,Running,1184
subcategoria,3552,16,Zapatillas Running,888
es_estrella,3552,2,False,2516


categoria
Running     1184
Fitness      888
Outdoor      888
Wellness     592
Name: count, dtype: int64
subcategoria
Zapatillas Running       888
Ropa Running             296
Zapatillas Trail         296
Ropa Montaña             296
Mancuernas Ajustables    148
Banco Gimnasio           148
Esterilla Fitness        148
Pesas Casa               148
Bandas Elásticas         148
Pesa Rusa                148
Bicicleta Montaña        148
Mochila Trekking         148
Esterilla Yoga           148
Bloque Yoga              148
Rodillera Yoga           148
Cojín Yoga               148
Name: count, dtype: int64


In [15]:
# Parse the date column and check its range and coverage
fecha_dt = pd.to_datetime(ventas_df["fecha"], errors="coerce")
print("Unparseable dates:", fecha_dt.isna().sum())
print("Range:", fecha_dt.min(), "->", fecha_dt.max())
print(fecha_dt.groupby(fecha_dt.dt.year).agg(["min", "max", "nunique"]))

# Check that every date has the same number of products and vice versa
print(ventas_df.groupby("fecha").size().describe())
print(ventas_df.groupby("producto_id").size().describe())

# Check for negative or zero values in the numeric columns
cols_num = ["precio_base", "unidades_vendidas", "precio_venta", "ingresos"]
print((ventas_df[cols_num] <= 0).sum())

# Check that ingresos equals unidades_vendidas * precio_venta
dif_ingresos = (ventas_df["unidades_vendidas"] * ventas_df["precio_venta"] - ventas_df["ingresos"]).abs()
print("Inconsistent ingresos:", (dif_ingresos > 0.011).sum())

# Check that product attributes are constant for each producto_id
attrs = ["nombre", "categoria", "subcategoria", "precio_base", "es_estrella"]
print(ventas_df.groupby("producto_id")[attrs].nunique().max())

# Ratio between precio_venta and precio_base
print((ventas_df["precio_venta"] / ventas_df["precio_base"]).describe())

Unparseable dates: 0
Range: 2021-10-25 00:00:00 -> 2024-11-30 00:00:00
             min        max  nunique
fecha                               
2021  2021-10-25 2021-11-30       37
2022  2022-10-25 2022-11-30       37
2023  2023-10-25 2023-11-30       37
2024  2024-10-25 2024-11-30       37
count    148.0
mean      24.0
std        0.0
min       24.0
25%       24.0
50%       24.0
75%       24.0
max       24.0
dtype: float64
count     24.0
mean     148.0
std        0.0
min      148.0
25%      148.0
50%      148.0
75%      148.0
max      148.0
dtype: float64
precio_base          0
unidades_vendidas    0
precio_venta         0
ingresos             0
dtype: int64
Inconsistent ingresos: 0
nombre          1
categoria       1
subcategoria    1
precio_base     1
es_estrella     1
dtype: int64
count    3552.000000
mean        0.989296
std         0.035626
min         0.850000
25%         0.978727
50%         0.995976
75%         1.013429
max         1.030000
dtype: float64


In [16]:
# Count global outliers with the IQR rule (1.5 * IQR)
for col in ["unidades_vendidas", "precio_venta", "ingresos"]:
    q1, q3 = ventas_df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((ventas_df[col] < q1 - 1.5 * iqr) | (ventas_df[col] > q3 + 1.5 * iqr)).sum()
    print(col, n_out, f"{n_out / len(ventas_df):.2%}")

# Count outliers in unidades_vendidas relative to each product (|z| > 4)
z_unidades = ventas_df.groupby("producto_id")["unidades_vendidas"].transform(lambda s: (s - s.mean()) / s.std())
outliers_df = ventas_df[z_unidades.abs() > 4]
print(len(outliers_df))

# Show in which month-day the product-level outliers happen
print(pd.to_datetime(outliers_df["fecha"]).dt.strftime("%m-%d").value_counts().sort_index())

unidades_vendidas 466 13.12%
precio_venta 296 8.33%
ingresos 405 11.40%
23
fecha
11-24    4
11-25    5
11-26    4
11-29    3
11-30    7
Name: count, dtype: int64


In [17]:
# Show shape, data types and non-null counts of the competitors' prices dataframe
print(competencia_df.shape)
competencia_df.info()

# Count null values per column
print(competencia_df.isna().sum())

# Count fully duplicated rows and duplicated (fecha, producto_id) keys
print("Full duplicates:", competencia_df.duplicated().sum())
print("Key duplicates:", competencia_df.duplicated(subset=["fecha", "producto_id"]).sum())

# Descriptive statistics of the competitors' prices
display(competencia_df.describe().T)

(3552, 5)
<class 'pandas.DataFrame'>
RangeIndex: 3552 entries, 0 to 3551
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   fecha         3552 non-null   str    
 1   producto_id   3552 non-null   str    
 2   Amazon        3552 non-null   float64
 3   Decathlon     3552 non-null   float64
 4   Deporvillage  3552 non-null   float64
dtypes: float64(3), str(2)
memory usage: 201.3 KB
fecha           0
producto_id     0
Amazon          0
Decathlon       0
Deporvillage    0
dtype: int64
Full duplicates: 0
Key duplicates: 0


,count,mean,std,min,25%,50%,75%,max
Amazon,3552.0,118.623407,156.095628,16.85,47.1175,73.180,114.3425,858.3500
Decathlon,3552.0,111.412182,148.508132,15.45,43.2850,66.285,111.1725,867.3375
Deporvillage,3552.0,118.894628,160.216448,16.77,47.3100,72.700,114.9850,932.3250


In [18]:
# Parse the date column and check its range and coverage
fecha_comp_dt = pd.to_datetime(competencia_df["fecha"], errors="coerce")
print("Unparseable dates:", fecha_comp_dt.isna().sum())
print(fecha_comp_dt.groupby(fecha_comp_dt.dt.year).agg(["min", "max", "nunique"]))

# Check that every date has the same number of products and vice versa
print(competencia_df.groupby("fecha").size().describe())
print(competencia_df.groupby("producto_id").size().describe())

# Check for negative or zero prices
cols_comp = ["Amazon", "Decathlon", "Deporvillage"]
print((competencia_df[cols_comp] <= 0).sum())

# Count outliers in each competitor's price with the IQR rule (1.5 * IQR)
for col in cols_comp:
    q1, q3 = competencia_df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((competencia_df[col] < q1 - 1.5 * iqr) | (competencia_df[col] > q3 + 1.5 * iqr)).sum()
    print(col, n_out, f"{n_out / len(competencia_df):.2%}")

# Check that both dataframes share exactly the same (fecha, producto_id) keys
claves_ventas = set(zip(ventas_df["fecha"], ventas_df["producto_id"]))
claves_comp = set(zip(competencia_df["fecha"], competencia_df["producto_id"]))
print("Only in ventas:", len(claves_ventas - claves_comp), "| Only in competencia:", len(claves_comp - claves_ventas))

# Compare each competitor's price against our own base and sale prices
cruce_df = ventas_df.merge(competencia_df, on=["fecha", "producto_id"], how="inner")
for col in cols_comp:
    print(col, "/ precio_base:", (cruce_df[col] / cruce_df["precio_base"]).describe()[["min", "mean", "max"]].round(3).to_dict())
    print(col, "/ precio_venta:", (cruce_df[col] / cruce_df["precio_venta"]).describe()[["min", "mean", "max"]].round(3).to_dict())

# Correlation between the competitors' prices
print(competencia_df[cols_comp].corr().round(3))

# Largest day-to-day price change per product and competitor
comp_ord_df = competencia_df.assign(fecha_dt=fecha_comp_dt).sort_values(["producto_id", "fecha_dt"])
for col in cols_comp:
    cambio = comp_ord_df.groupby("producto_id")[col].pct_change().abs()
    print(col, "max change:", round(cambio.max(), 3), "| changes > 30%:", (cambio > 0.3).sum())

Unparseable dates: 0
             min        max  nunique
fecha                               
2021  2021-10-25 2021-11-30       37
2022  2022-10-25 2022-11-30       37
2023  2023-10-25 2023-11-30       37
2024  2024-10-25 2024-11-30       37
count    148.0
mean      24.0
std        0.0
min       24.0
25%       24.0
50%       24.0
75%       24.0
max       24.0
dtype: float64
count     24.0
mean     148.0
std        0.0
min      148.0
25%      148.0
50%      148.0
75%      148.0
max      148.0
dtype: float64
Amazon          0
Decathlon       0
Deporvillage    0
dtype: int64
Amazon 296 8.33%
Decathlon 296 8.33%
Deporvillage 296 8.33%
Only in ventas: 0 | Only in competencia: 0
Amazon / precio_base: {'min': 0.667, 'mean': 0.98, 'max': 1.198}
Amazon / precio_venta: {'min': 0.674, 'mean': 0.991, 'max': 1.287}
Decathlon / precio_base: {'min': 0.73, 'mean': 0.912, 'max': 1.152}
Decathlon / precio_venta: {'min': 0.715, 'mean': 0.923, 'max': 1.219}
Deporvillage / precio_base: {'min': 0.765, 'mea

In [20]:
# Convert the date column of the sales dataframe to datetime
ventas_df["fecha"] = pd.to_datetime(ventas_df["fecha"])

# Convert the date column of the competitors' prices dataframe to datetime
competencia_df["fecha"] = pd.to_datetime(competencia_df["fecha"])

# Print the resulting dtype and first values of the date column in both dataframes
print(ventas_df["fecha"].dtype)
print(ventas_df["fecha"].head())
print(competencia_df["fecha"].dtype)
print(competencia_df["fecha"].head())

datetime64[us]
0   2021-10-25
1   2021-10-25
2   2021-10-25
3   2021-10-25
4   2021-10-25
Name: fecha, dtype: datetime64[us]
datetime64[us]
0   2021-10-25
1   2021-10-25
2   2021-10-25
3   2021-10-25
4   2021-10-25
Name: fecha, dtype: datetime64[us]


In [23]:
# Merge sales and competitors' prices using fecha and producto_id as keys (one row per key in both dataframes)
df = ventas_df.merge(competencia_df, on=["fecha", "producto_id"], how="left", validate="one_to_one")

# Check the shape and the first rows of the merged dataframe
print(df.shape)
df.head()

(3552, 13)


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2021-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,6,118.36,710.16,82.96,111.88,97.43
1,2021-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,10,136.82,1368.20,112.56,108.61,115.58
2,2021-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,2,84.93,169.86,79.79,78.44,80.11
3,2021-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,2,75.42,150.84,72.60,67.29,74.45
4,2021-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,2,35.87,71.74,37.71,33.60,33.07
